# Présentation des données

Tour d'horizon des types de données disponibles pour le projet, pour deux sources :

1. **NHL** (API officielle, chargée dans MySQL par `scripts/download_nhl_data.py`) :
   matchs, statistiques d'équipes, de patineurs et de gardiens, alignements ;
2. **MoneyPuck** (fichiers CSV *game by game*) : statistiques avancées (buts attendus xG,
   chances par niveau de danger, etc.) par équipe, patineur et gardien.

Pour chaque type : granularité (une ligne = quoi), volume, variables disponibles et un exemple
sur **un même match**, puis une comparaison des deux sources sur ce match.

In [1]:
import pandas as pd
from sqlalchemy import text

from nhl_forecast.config import PROJECT_ROOT, get_settings
from nhl_forecast.data.db.mysql import make_engine
from nhl_forecast.data.nhl_collect import stats_table
from nhl_forecast.data.sources.nhl_api import REPORTS

In [2]:
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

engine = make_engine()
DB = get_settings().mysql_database
MP_DIR = PROJECT_ROOT / "data" / "raw" / "moneypuck"

GAME_ID = 2025020001  # match d'exemple (premier match de la saison 2025-2026)
SEASON_ID = "20252026"  # convention NHL ; MoneyPuck note la même saison « 2025 »
FIRST_SEASON = "20052006"


def sql(query, **params):
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

## 1. NHL (base de données)

### 1.1 Matchs — table `games`

Une ligne par match : date, équipes locale/visiteuse, score final et façon dont le match s'est
terminé (`REG`, `OT`, `SO`). C'est la table qui fournit la **variable cible** (victoire locale).

In [3]:
games = sql(
    """
    SELECT g.game_id, g.season_id, g.game_type, g.date, g.time,
           th.team_name AS home_team, ta.team_name AS away_team,
           g.score_home, g.score_away, g.end_game_type
    FROM games g
    JOIN teams th ON th.team_id = g.home
    JOIN teams ta ON ta.team_id = g.away
    WHERE g.league = 'nhl' AND g.season_id >= :first AND g.status = 'OFF'
      AND g.game_type IN ('R', 'P')
    """,
    first=FIRST_SEASON,
)
print(f"{len(games):,} matchs terminés depuis {FIRST_SEASON}")
games[games["game_id"] == str(GAME_ID)]

25,506 matchs terminés depuis 20052006


,game_id,season_id,game_type,date,time,home_team,away_team,score_home,score_away,end_game_type
24087,2025020001,20252026,R,2025-10-07,17:00:00,Panthers,Blackhawks,3,2,REG


In [4]:
games["home_win"] = games["score_home"] > games["score_away"]
par_saison = (
    games[games["game_type"] == "R"]
    .groupby("season_id")
    .agg(
        matchs=("game_id", "size"),
        victoires_locales=("home_win", "mean"),
        prolongation=("end_game_type", lambda s: (s == "OT").mean()),
        fusillade=("end_game_type", lambda s: (s == "SO").mean()),
    )
)
par_saison.style.format({c: "{:.1%}" for c in par_saison.columns[1:]})

,matchs,victoires_locales,prolongation,fusillade
season_id,,,,
20052006,1230,57.4%,11.1%,11.8%
20062007,1230,55.0%,9.5%,13.3%
20072008,1230,53.7%,9.4%,12.7%
20082009,1230,55.9%,10.0%,12.9%
20092010,1230,56.1%,9.5%,15.0%
20102011,1230,51.9%,12.0%,12.1%
20112012,1230,55.9%,9.7%,14.7%
20122013,720,56.8%,9.0%,13.5%
20132014,1230,53.7%,10.5%,14.5%


Saisons écourtées : 2012-2013 (lock-out, 48 matchs par équipe), 2019-2020 (interrompue en
mars 2020) et 2020-2021 (56 matchs). L'avantage de la glace reste autour de 53-55 %.

### 1.2 Inventaire des rapports statistiques

Les statistiques d'équipes (`t_*`), de patineurs (sans préfixe) et de gardiens (`g_*`) viennent
des rapports de nhl.com/stats, demandés **par match** : une ligne par équipe-match ou par
joueur-match. Nombre de lignes approximatif (`information_schema`).

In [5]:
tables = sql(
    """
    SELECT t.TABLE_NAME AS table_name, t.TABLE_ROWS AS lignes_approx,
           COUNT(c.COLUMN_NAME) AS colonnes
    FROM information_schema.TABLES t
    JOIN information_schema.COLUMNS c
      ON c.TABLE_SCHEMA = t.TABLE_SCHEMA AND c.TABLE_NAME = t.TABLE_NAME
    WHERE t.TABLE_SCHEMA = :db
    GROUP BY t.TABLE_NAME, t.TABLE_ROWS
    """,
    db=DB,
).set_index("table_name")

inventaire = pd.DataFrame(
    [
        {"entité": entity, "rapport": report, "table": stats_table(entity, report)}
        for entity, reports in REPORTS.items()
        for report in reports
    ]
).join(tables, on="table")
inventaire.groupby("entité").agg(
    rapports=("table", "size"),
    colonnes=("colonnes", "sum"),
    lignes_approx=("lignes_approx", "sum"),
)

,rapports,colonnes,lignes_approx
entité,,,
goalie,7,182,408239.0
skater,17,462,18803963.0
team,21,543,1426672.0


In [6]:
inventaire

,entité,rapport,table,lignes_approx,colonnes
0,team,summary,t_summary,74233.0,27
1,team,daysbetweengames,t_daysbetweengames,77321.0,25
2,team,faceoffpercentages,t_faceoffpercentages,63676.0,21
3,team,faceoffwins,t_faceoffwins,61995.0,29
4,team,goalgames,t_goalgames,78169.0,23
5,team,goalsagainstbystrength,t_goalsagainstbystrength,68224.0,32
6,team,goalsbyperiod,t_goalsbyperiod,77596.0,26
7,team,goalsforbystrength,t_goalsforbystrength,38729.0,32
8,team,leadingtrailing,t_leadingtrailing,74503.0,32
9,team,outshootoutshotby,t_outshootoutshotby,75188.0,28


### 1.3 Statistiques d'équipes — ex. `t_summary`

Une ligne par équipe et par match (donc deux lignes par match). Le rapport `summary` contient
les résultats de base ; les autres rapports détaillent avantage numérique, mises en jeu,
tirs, buts par période, etc.

In [7]:
t_summary = sql("SELECT * FROM t_summary WHERE gameId = :g", g=GAME_ID)
t_summary.set_index("teamFullName").T

teamFullName,Florida Panthers,Chicago Blackhawks
faceoffWinPct,0.585714,0.414285
gameDate,2025-10-07,2025-10-07
gameId,2025020001,2025020001
gamesPlayed,1,1
goalsAgainst,2,3
goalsAgainstPerGame,2,3
goalsFor,3,2
goalsForPerGame,3,2
homeRoad,H,R
losses,0,1


In [8]:
# Quelques autres rapports d'équipe pour le même match
for table in ["t_powerplay", "t_realtime", "t_summaryshooting"]:
    df = sql(f"SELECT * FROM {table} WHERE gameId = :g", g=GAME_ID)
    print(f"--- {table} : {df.shape[1]} colonnes")
    display(df.set_index("teamFullName").T.head(15))

--- t_powerplay : 24 colonnes


teamFullName,Florida Panthers,Chicago Blackhawks
gameDate,2025-10-07,2025-10-07
gameId,2025020001,2025020001
gamesPlayed,1,1
homeRoad,H,R
losses,0,1
opponentTeamAbbrev,CHI,FLA
otLosses,0,0
pointPct,1.0,0.0
points,2,0
powerPlayGoalsFor,1,0


--- t_realtime : 19 colonnes


teamFullName,Florida Panthers,Chicago Blackhawks
blockedShots,15,16
blockedShotsPer60,14.9976,15.9984
emptyNetGoals,0,0
gameDate,2025-10-07,2025-10-07
gameId,2025020001,2025020001
gamesPlayed,1,1
giveaways,17,19
giveawaysPer60,16.9992,18.9972
hits,28,27
hitsPer60,27.9972,27.0


--- t_summaryshooting : 22 colonnes


teamFullName,Florida Panthers,Chicago Blackhawks
gameDate,2025-10-07,2025-10-07
gameId,2025020001,2025020001
gamesPlayed,1,1
homeRoad,H,R
opponentTeamAbbrev,CHI,FLA
satAgainst,41,58
satAhead,7,-2
satBehind,2,-7
satClose,14,-14
satFor,58,41


### 1.4 Statistiques des patineurs — ex. `summary`, `timeonice`

Une ligne par patineur et par match. 17 rapports, joignables sur (`gameId`, `playerId`).

In [9]:
skaters = sql(
    """
    SELECT s.teamAbbrev, s.skaterFullName, s.positionCode, s.goals, s.assists, s.points,
           s.plusMinus, s.shots, s.penaltyMinutes, t.timeOnIce, t.ppTimeOnIce, t.shifts
    FROM summary s
    JOIN timeonice t ON t.gameId = s.gameId AND t.playerId = s.playerId
    WHERE s.gameId = :g
    ORDER BY s.teamAbbrev, s.points DESC, t.timeOnIce DESC
    """,
    g=GAME_ID,
)
skaters["timeOnIce"] = (skaters["timeOnIce"] / 60).round(1)  # secondes -> minutes
skaters["ppTimeOnIce"] = (skaters["ppTimeOnIce"] / 60).round(1)
print(f"{len(skaters)} patineurs")
skaters.head(12)

36 patineurs


,teamAbbrev,skaterFullName,positionCode,goals,assists,points,plusMinus,shots,penaltyMinutes,timeOnIce,ppTimeOnIce,shifts
0,CHI,Frank Nazar,C,1,1,2,1,1,0,20.2,4.0,27
1,CHI,Tyler Bertuzzi,L,0,2,2,1,1,0,19.2,4.0,24
2,CHI,Teuvo Teravainen,C,1,1,2,1,1,0,18.1,4.0,26
3,CHI,Sam Rinzel,D,0,0,0,0,1,0,25.3,4.3,29
4,CHI,Connor Bedard,C,0,0,0,0,2,0,22.0,4.1,26
5,CHI,Connor Murphy,D,0,0,0,-1,2,0,18.6,0.0,25
6,CHI,Matt Grzelcyk,D,0,0,0,0,2,0,18.4,0.0,25
7,CHI,Ethan Del Mastro,D,0,0,0,-1,1,0,17.6,0.0,25
8,CHI,Ilya Mikheyev,R,0,0,0,0,1,0,17.1,0.1,24
9,CHI,Artyom Levshunov,D,0,0,0,1,2,4,16.6,1.7,23


### 1.5 Statistiques des gardiens — ex. `g_summary`, `g_savesbystrength`

Une ligne par gardien ayant joué le match (7 rapports).

In [10]:
goalies = sql(
    """
    SELECT s.teamAbbrev, s.goalieFullName, s.gamesStarted, s.timeOnIce, s.shotsAgainst,
           s.saves, s.goalsAgainst, s.savePct, s.wins, s.losses, s.otLosses,
           v.evShotsAgainst, v.evSaves, v.ppShotsAgainst, v.ppSaves, v.shShotsAgainst, v.shSaves
    FROM g_summary s
    JOIN g_savesbystrength v ON v.gameId = s.gameId AND v.playerId = s.playerId
    WHERE s.gameId = :g
    """,
    g=GAME_ID,
)
goalies["timeOnIce"] = (goalies["timeOnIce"] / 60).round(1)
goalies

,teamAbbrev,goalieFullName,gamesStarted,timeOnIce,shotsAgainst,saves,goalsAgainst,savePct,wins,losses,otLosses,evShotsAgainst,evSaves,ppShotsAgainst,ppSaves,shShotsAgainst,shSaves
0,FLA,Sergei Bobrovsky,1,60.0,19,17,2,0.89473,1,0,0,17,15,2,2,0,0
1,CHI,Spencer Knight,1,59.1,37,34,3,0.91891,0,1,0,31,29,5,4,1,1


### 1.6 Alignements — table `rosters`

Une ligne par joueur habillé (`position` : `g` gardien, `p` patineur). Sert à savoir **avant le
match** quels joueurs sont présents, donc à agréger les statistiques individuelles au niveau de
l'équipe.

In [12]:
roster = sql(
    """
    SELECT t.team_name, r.player_id, r.name, r.position, r.position_long, r.scratched
    FROM rosters r JOIN teams t ON t.team_id = r.team_id
    WHERE r.game_id = :g
    """,
    g=GAME_ID,
)
display(pd.crosstab(roster["team_name"], roster["position_long"]))
roster.head(10)

position_long,C,D,G,L,R
team_name,,,,,
Blackhawks,8,6,2,3,1
Panthers,8,6,2,3,1


,team_name,player_id,name,position,position_long,scratched
0,Panthers,8473419,B. Marchand,p,L,0
1,Blackhawks,8473422,N. Foligno,p,L,0
2,Panthers,8473507,J. Petry,p,D,0
3,Panthers,8475179,D. Kulikov,p,D,0
4,Panthers,8475683,S. Bobrovsky,g,G,0
5,Blackhawks,8476473,C. Murphy,p,D,0
6,Blackhawks,8476882,T. Teravainen,p,C,0
7,Blackhawks,8476891,M. Grzelcyk,p,D,0
8,Panthers,8477409,C. Verhaeghe,p,C,0
9,Blackhawks,8477444,A. Burakovsky,p,L,0


## 2. MoneyPuck (CSV)

Fichiers *game by game* de moneypuck.com (depuis 2008-2009). Différence majeure avec la NHL :
chaque joueur (ou équipe) a **une ligne par situation de jeu** pour chaque match :

| `situation` | Signification |
|---|---|
| `all` | toutes situations |
| `5on5` | égalité numérique à 5 contre 5 |
| `5on4` | avantage numérique |
| `4on5` | désavantage numérique |
| `other` | autres (4 c. 4, 3 c. 3, filet désert…) |

Les variables reposent sur le modèle de **buts attendus (xG)** de MoneyPuck (probabilité qu'un
tir devienne un but selon distance, angle, type de tir, rebond…). Pour la présentation, on se limite
à **une saison régulière (2025-2026)** pour les trois entités.

Source : [moneypuck.com/data.htm](https://moneypuck.com/data.htm), section *Game By Game Level
Data*. Les équipes sont dans un seul fichier toutes saisons confondues (`all_teams.csv`, séries
incluses, colonne `playoffGame`) ; les patineurs et gardiens sont publiés par saison
(`2008_to_2024`, puis un fichier par saison, saison régulière seulement). Le fichier d'équipes
est donc filtré pour suivre le même standard.

In [13]:
MP_SEASON = 2025  # saison 2025-2026 (MoneyPuck note l'année de début)

mp_skaters = pd.read_csv(MP_DIR / "skaters" / f"{MP_SEASON}.csv")
mp_goalies = pd.read_csv(MP_DIR / "goalies" / f"{MP_SEASON}.csv")

# Fichier d'équipes : toutes saisons et séries -> filtré sur la même saison régulière
teams_csv = MP_DIR / "teams" / "all_teams.csv"
mp_teams = pd.read_csv(teams_csv).query("season == @MP_SEASON and playoffGame == 0")

for name, df in [("équipes", mp_teams), ("patineurs", mp_skaters), ("gardiens", mp_goalies)]:
    print(
        f"{name:10s} {df.shape[0]:>10,} lignes x {df.shape[1]:>3} colonnes"
        f" | saisons {df['season'].min()}-{df['season'].max()}"
        f" | situations {sorted(df['situation'].unique())}"
    )

équipes        13,120 lignes x 111 colonnes | saisons 2025-2025 | situations ['4on5', '5on4', '5on5', 'all', 'other']
patineurs     236,060 lignes x 157 colonnes | saisons 2025-2025 | situations ['4on5', '5on4', '5on5', 'all', 'other']
gardiens       13,830 lignes x  39 colonnes | saisons 2025-2025 | situations ['4on5', '5on4', '5on5', 'all', 'other']


### 2.1 Matchs (dérivés du fichier d'équipes)

MoneyPuck ne publie pas de table de matchs séparée : on la reconstruit à partir des lignes
d'équipe locale en situation `all`. Les abréviations diffèrent de la NHL (`T.B`, `N.J`, `S.J`,
`L.A`) ; la jointure avec la NHL se fait donc sur `gameId`.

In [14]:
mp_games = (
    mp_teams.query("situation == 'all' and home_or_away == 'HOME'")
    .loc[
        :,
        [
            "gameId",
            "season",
            "gameDate",
            "playoffGame",
            "playerTeam",
            "opposingTeam",
            "goalsFor",
            "goalsAgainst",
            "xGoalsFor",
            "xGoalsAgainst",
            "shotsOnGoalFor",
            "shotsOnGoalAgainst",
        ],
    ]
    .rename(columns={"playerTeam": "home", "opposingTeam": "away"})
)
print(f"{len(mp_games):,} matchs")
mp_games[mp_games["gameId"] == GAME_ID]

1,312 matchs


,gameId,season,gameDate,playoffGame,home,away,goalsFor,goalsAgainst,xGoalsFor,xGoalsAgainst,shotsOnGoalFor,shotsOnGoalAgainst
123481,2025020001,2025,20251007,0,FLA,CHI,3.0,2.0,3.882,1.508,37.0,19.0


### 2.2 Statistiques d'équipes

Une ligne par équipe × match × situation. Les variables sont symétriques (`...For` /
`...Against`) : tirs, xG, chances par niveau de danger, rebonds, mises en jeu, mises en échec…

In [15]:
mp_teams.query("gameId == @GAME_ID").set_index(["playerTeam", "situation"]).T.head(40)

playerTeam                                FLA                                                         CHI                                                
situation                               other         all        5on5        4on5        5on4       other         all        5on5        4on5        5on4
team                                      FLA         FLA         FLA         FLA         FLA         CHI         CHI         CHI         CHI         CHI
season                                   2025        2025        2025        2025        2025        2025        2025        2025        2025        2025
name                                      FLA         FLA         FLA         FLA         FLA         CHI         CHI         CHI         CHI         CHI
gameId                             2025020001  2025020001  2025020001  2025020001  2025020001  2025020001  2025020001  2025020001  2025020001  2025020001
opposingTeam                              CHI         CHI         CHI         CHI         CHI         FLA         FLA         FLA         FLA         FLA
home_or_away                             HOME        HOME        HOME        HOME        HOME        AWAY        AWAY        AWAY        AWAY        AWAY
gameDate                             20251007    20251007    20251007    20251007    20251007    20251007    20251007    20251007    20251007    20251007
position                           Team Level  Team Level  Team Level  Team Level  Team Level  Team Level  Team Level  Team Level  Team Level  Team Level
xGoalsPercentage                          0.0      0.7202      0.6824      0.8902         1.0         1.0      0.2798      0.3176         0.0      0.1098
corsiPercentage                           0.0      0.5798       0.596      0.2222         1.0         1.0      0.4202       0.404         0.0      0.7778
fenwickPercentage                         0.0      0.6098      0.6087         0.4         1.0         1.0      0.3902      0.3913         0.0         0.6
iceTime                                  56.0      3600.0      2962.0       360.0       222.0        56.0      3600.0      2962.0       222.0       360.0
xOnGoalFor                                0.0      38.161      31.868       1.701       4.591       1.049      23.487      20.192         0.0       2.247
xGoalsFor                                 0.0       3.882         3.0       0.235       0.648       0.084       1.508       1.396         0.0       0.029
xReboundsFor                              0.0       2.824       2.321        0.12       0.382       0.114       1.352       1.051         0.0       0.187
xFreezeFor                                0.0       8.882       7.815         0.2       0.867       0.247       5.884       5.138         0.0       0.499
xPlayStoppedFor                           0.0        1.19       1.008       0.038       0.144       0.074       0.768       0.626         0.0       0.068
xPlayContinuedInZoneFor                   0.0      19.618      16.508       0.461        2.65       0.994      12.841      10.539         0.0       1.307
xPlayContinuedOutsideZoneFor              0.0      13.603      11.348       0.946       1.309       0.488       9.647       8.249         0.0        0.91
flurryAdjustedxGoalsFor                   0.0       3.807       2.956       0.235       0.616       0.083       1.501        1.39         0.0       0.029
scoreVenueAdjustedxGoalsFor               0.0       3.804       2.922       0.235       0.648       0.084       1.558       1.446         0.0       0.029
flurryScoreVenueAdjustedxGoalsFor         0.0       3.731       2.879       0.235       0.616       0.083       1.551       1.439         0.0       0.029
shotsOnGoalFor                            0.0        37.0        31.0         1.0         5.0         0.0        19.0        17.0         0.0         2.0
missedShotsFor                            0.0        13.0        11.0         1.0         1.0         2.0        13.0        10.0         0.0         

In [16]:
def groupes(columns, prefixes):
    """Nombre de colonnes par préfixe (pour résumer les fichiers très larges)."""
    out = {p: sum(c.startswith(p) for c in columns) for p in prefixes}
    out["autres"] = len(columns) - sum(out.values())
    return pd.Series(out, name="colonnes")


groupes(mp_teams.columns, ["x", "lowDanger", "mediumDanger", "highDanger"])

x               19
lowDanger        6
mediumDanger     6
highDanger       6
autres          74
Name: colonnes, dtype: int64

### 2.3 Statistiques des patineurs

Une ligne par patineur × match × situation (~150 colonnes). Préfixes :

- `I_F_` : production **individuelle** du joueur (tirs, xG, points, mises en échec…) ;
- `OnIce_F_` / `OnIce_A_` : ce que **son équipe** génère / concède quand il est sur la glace ;
- `OffIce_` : idem quand il est au banc (pour des mesures relatives).

In [17]:
groupes(mp_skaters.columns, ["I_F_", "OnIce_F_", "OnIce_A_", "OffIce_"])

I_F_        59
OnIce_F_    30
OnIce_A_    30
OffIce_      4
autres      34
Name: colonnes, dtype: int64

In [18]:
cols = [
    "playerTeam",
    "name",
    "position",
    "icetime",
    "I_F_goals",
    "I_F_points",
    "I_F_shotsOnGoal",
    "I_F_xGoals",
    "OnIce_F_xGoals",
    "OnIce_A_xGoals",
    "onIce_xGoalsPercentage",
    "gameScore",
]
ex = mp_skaters.query("gameId == @GAME_ID and situation == 'all'")[cols].copy()
ex["icetime"] = (ex["icetime"] / 60).round(1)  # secondes -> minutes
ex.sort_values(["playerTeam", "I_F_points", "icetime"], ascending=[True, False, False]).head(12)

,playerTeam,name,position,icetime,I_F_goals,I_F_points,I_F_shotsOnGoal,I_F_xGoals,OnIce_F_xGoals,OnIce_A_xGoals,onIce_xGoalsPercentage,gameScore
49651,CHI,Frank Nazar,C,20.2,1.0,2.0,1.0,0.140,0.477,2.063,0.1878,1.105
46161,CHI,Tyler Bertuzzi,L,19.2,0.0,2.0,1.0,0.116,0.477,1.180,0.2879,1.165
49976,CHI,Teuvo Teravainen,C,18.1,1.0,2.0,1.0,0.043,0.546,1.764,0.2364,1.675
45556,CHI,Sam Rinzel,D,25.3,0.0,0.0,1.0,0.084,0.573,1.988,0.2237,-0.125
50721,CHI,Connor Bedard,C,22.0,0.0,0.0,2.0,0.229,0.580,1.308,0.3072,-0.560
47471,CHI,Connor Murphy,D,18.6,0.0,0.0,2.0,0.060,0.371,1.304,0.2215,-0.400
45821,CHI,Matt Grzelcyk,D,18.4,0.0,0.0,2.0,0.090,0.508,1.284,0.2835,-0.300
44796,CHI,Ethan Del Mastro,D,17.6,0.0,0.0,1.0,0.010,0.476,1.212,0.2820,-0.525
49271,CHI,Ilya Mikheyev,R,17.1,0.0,0.0,1.0,0.066,0.176,0.704,0.2000,-0.475
44886,CHI,Wyatt Kaiser,D,16.6,0.0,0.0,2.0,0.255,0.517,1.152,0.3098,0.600


### 2.4 Statistiques des gardiens

Une ligne par gardien × match × situation. Variables clés : buts accordés vs buts attendus
(`goals` vs `xGoals`, d'où la mesure *goals saved above expected*), répartition des tirs par
niveau de danger, rebonds accordés vs attendus.

In [19]:
cols = [
    "playerTeam",
    "name",
    "situation",
    "icetime",
    "unblocked_shot_attempts",
    "ongoal",
    "goals",
    "xGoals",
    "lowDangerShots",
    "mediumDangerShots",
    "highDangerShots",
    "rebounds",
    "xRebounds",
]
g = mp_goalies.query("gameId == @GAME_ID")[cols].copy()
g["icetime"] = (g["icetime"] / 60).round(1)
g["GSAx"] = g["xGoals"] - g["goals"]  # buts sauvés au-delà des attentes
g.sort_values(["playerTeam", "situation"])

,playerTeam,name,situation,icetime,unblocked_shot_attempts,ongoal,goals,xGoals,lowDangerShots,mediumDangerShots,highDangerShots,rebounds,xRebounds,GSAx
2723,CHI,Spencer Knight,4on5,3.7,8.0,5.0,1.0,0.648,3.0,2.0,1.0,1.0,0.382,-0.352
2724,CHI,Spencer Knight,5on4,6.0,2.0,1.0,0.0,0.235,1.0,0.0,1.0,0.0,0.120,0.235
2722,CHI,Spencer Knight,5on5,49.4,59.0,31.0,2.0,3.000,28.0,11.0,3.0,1.0,2.321,1.000
2721,CHI,Spencer Knight,all,59.1,69.0,37.0,3.0,3.882,32.0,13.0,5.0,2.0,2.824,0.882
2720,CHI,Spencer Knight,other,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,0.000
4933,FLA,Sergei Bobrovsky,4on5,6.0,7.0,2.0,0.0,0.029,3.0,0.0,0.0,0.0,0.187,0.029
4934,FLA,Sergei Bobrovsky,5on4,3.7,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,0.000
4932,FLA,Sergei Bobrovsky,5on5,49.4,40.0,17.0,2.0,1.396,21.0,5.0,1.0,1.0,1.051,-0.604
4931,FLA,Sergei Bobrovsky,all,60.0,50.0,19.0,2.0,1.508,26.0,5.0,1.0,1.0,1.352,-0.492
4930,FLA,Sergei Bobrovsky,other,0.9,3.0,0.0,0.0,0.084,2.0,0.0,0.0,0.0,0.114,0.084


## 3. Comparaison NHL vs MoneyPuck sur le match d'exemple

Les deux sources décrivent le même match ; les comptes de base (buts, tirs) devraient
concorder, alors que MoneyPuck ajoute des variables dérivées (xG) absentes de la NHL.

In [20]:
nhl_side = t_summary.assign(home_or_away=t_summary["homeRoad"].map({"H": "HOME", "R": "AWAY"}))[
    ["home_or_away", "teamFullName", "goalsFor", "goalsAgainst", "shotsForPerGame"]
]

mp_side = mp_teams.query("gameId == @GAME_ID and situation == 'all'")[
    ["home_or_away", "playerTeam", "goalsFor", "goalsAgainst", "shotsOnGoalFor", "xGoalsFor"]
]

nhl_side.merge(mp_side, on="home_or_away", suffixes=("_nhl", "_mp")).set_index("home_or_away").T

home_or_away,HOME,AWAY
teamFullName,Florida Panthers,Chicago Blackhawks
goalsFor_nhl,3,2
goalsAgainst_nhl,2,3
shotsForPerGame,37,19
playerTeam,FLA,CHI
goalsFor_mp,3.0,2.0
goalsAgainst_mp,2.0,3.0
shotsOnGoalFor,37.0,19.0
xGoalsFor,3.882,1.508


## 4. Synthèse

| | NHL (API officielle) | MoneyPuck |
|---|---|---|
| Accès | API JSON → MySQL | CSV téléchargeables |
| Période | depuis 2005-2006 (ici) | depuis 2008-2009 |
| Matchs | table `games` (score, OT/SO) | dérivés du fichier d'équipes |
| Granularité | équipe-match, joueur-match | équipe/joueur × match × **situation** |
| Équipes | 21 rapports `t_*` | 1 fichier, ~110 colonnes For/Against |
| Patineurs | 17 rapports (comptes, temps de glace, AN/DN…) | 1 fichier, ~150 colonnes (individuel, sur glace, hors glace) |
| Gardiens | 7 rapports `g_*` | 1 fichier, ~40 colonnes |
| Alignements | `rosters` (joueurs habillés) | implicites (joueurs ayant joué) |
| Points forts | officiel, complet, découpage par force | xG, niveaux de danger, ajustements score/lieu |
| Limites | API non documentée, pas de xG | modèle propriétaire, abréviations propres |

**Attention aux fuites d'information** : toutes ces statistiques décrivent ce qui s'est passé
*pendant* le match. Pour prédire un match, il faut les agréger sur les matchs **précédents**
(moyennes de saison, fenêtres glissantes, momentum).